# 12 — Rotate BCHH seismograms and compute polarization time series

This notebook extends the canonical Notebook 02 products. It:

1. loads the corrected three-component BCHH seismogram and event-epoch StationXML recorded in `02_analysis_configuration.json`;
2. verifies/rotates the seismic components to Z/N/E using ObsPy metadata;
3. rotates N/E to radial/transverse using the SLC-40 back-azimuth recorded by Notebook 02;
4. computes sliding-window covariance linearity, rectilinearity, and planarity;
5. computes complex-signal ellipticity and degree of polarization with TwistPy;
6. writes reusable CSV, NPZ, Pickle, JSON, and diagnostic-figure products for Notebook 13.

The real-covariance attributes and TwistPy ellipticity are kept distinct because they are based on different polarization representations.


## 1. Imports, project paths, and authoritative inputs


In [1]:
from __future__ import annotations

import json
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from obspy import Stream, Trace, UTCDateTime, read, read_inventory
from obspy.signal.rotate import rotate_ne_rt

try:
    from twistpy.polarization import TimeDomainAnalysis3C
except ImportError as exc:
    raise ImportError(
        "TwistPy is required for Notebook 12. Install it in this kernel with "
        "`python -m pip install twistpy`."
    ) from exc

project_root = Path.cwd().resolve()
if project_root.name == "notebooks2":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from modules import project_config as config

ANALYSIS_CONFIG_FILE = config.OUTPUT_DIR / "02_analysis_configuration.json"
if not ANALYSIS_CONFIG_FILE.exists():
    raise FileNotFoundError(
        f"Run 02_prepare_analysis_inputs.ipynb first: {ANALYSIS_CONFIG_FILE}"
    )

analysis_config = json.loads(ANALYSIS_CONFIG_FILE.read_text())

SOURCE_STREAM_FILE = Path(
    analysis_config["baseline_removed_streams"]["pickle"]
).expanduser()
STATIONXML_FILE = Path(
    analysis_config["notebook_01_products"]["calibrated_stationxml"]
).expanduser()

for label, path in {
    "corrected waveform Stream": SOURCE_STREAM_FILE,
    "calibrated StationXML": STATIONXML_FILE,
}.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")

OUTPUT_ROOT = config.OUTPUT_DIR / "12_polarization"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
config.FIGURE_DIR.mkdir(parents=True, exist_ok=True)

POLARIZATION_CSV = OUTPUT_ROOT / "12_polarization_timeseries.csv"
POLARIZATION_NPZ = OUTPUT_ROOT / "12_polarization_timeseries.npz"
ROTATED_ZNE_PICKLE = OUTPUT_ROOT / "12_seismic_zne.pkl"
ROTATED_ZRT_PICKLE = OUTPUT_ROOT / "12_seismic_zrt.pkl"
SUMMARY_JSON = OUTPUT_ROOT / "12_polarization_summary.json"
DIAGNOSTIC_FIGURE = config.FIGURE_DIR / "12_polarization_diagnostics.png"

print("Waveform source:", SOURCE_STREAM_FILE)
print("StationXML:", STATIONXML_FILE)
print("Output directory:", OUTPUT_ROOT)


/Users/thompsong/Developer/TwistPy/twistpy/polarization/time.py:95: SyntaxWarning: invalid escape sequence '\s'
  P^2=\sum_{j,k=1}^{n}(\lambda_j-\lambda_k)^2/[2(n-1)(\sum_{j=1}^{n}(\lambda_j)^2)]
/Users/thompsong/Developer/TwistPy/twistpy/polarization/time.py:708: SyntaxWarning: invalid escape sequence '\s'
  P^2=\sum_{j,k=1}^{n}(\lambda_j-\lambda_k)^2/[2(n-1)(\sum_{j=1}^{n}(\lambda_j)^2)]


Waveform source: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/derived2/bchh_corrected_moving_median_baseline_removed.pkl
StationXML: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/outputs2/01_BCHH_20160901_empirically_calibrated.xml
Output directory: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/outputs2/12_polarization


## 2. Analysis controls


In [2]:
# Broad enough to retain the impulsive ground-coupled airwave and Rayleigh-wave
# energy while suppressing drift and high-frequency noise. Adjust after review.
FILTER_FREQMIN_HZ = 0.5
FILTER_FREQMAX_HZ = 20.0
FILTER_CORNERS = 4
FILTER_ZEROPHASE = True

# TwistPy and covariance window controls.
WINDOW_LENGTH_S = 0.50
WINDOW_OVERLAP = 0.90

# Diagnostic display interval around the adopted explosion time.
PLOT_START = config.EXPLOSION_TIME - 5.0
PLOT_END = config.EXPLOSION_TIME + 35.0

# Leave None to analyze the complete canonical Notebook 02 time window.
ANALYSIS_START = None
ANALYSIS_END = None

if not (0.0 <= WINDOW_OVERLAP < 1.0):
    raise ValueError("WINDOW_OVERLAP must be in [0, 1).")


## 3. Load, isolate, and rotate the three-component seismogram


In [3]:
st_all = read(str(SOURCE_STREAM_FILE), format="PICKLE")
inventory = read_inventory(str(STATIONXML_FILE))

seismic_channels = tuple(
    str(channel).upper() for channel in analysis_config["seismic_channels"]
)
st_seismic = Stream(
    trace.copy() for trace in st_all
    if trace.stats.channel.upper() in seismic_channels
)

if len(st_seismic) != 3:
    raise ValueError(
        "Expected exactly three canonical seismic components from Notebook 02; "
        f"found {[trace.id for trace in st_seismic]}"
    )

if ANALYSIS_START is not None or ANALYSIS_END is not None:
    st_seismic.trim(
        UTCDateTime(ANALYSIS_START) if ANALYSIS_START is not None else st_seismic[0].stats.starttime,
        UTCDateTime(ANALYSIS_END) if ANALYSIS_END is not None else st_seismic[0].stats.endtime,
        pad=False,
    )

# First try the channels exactly as written. If they are not already Z/N/E,
# use the event-epoch StationXML orientation metadata to rotate them.
components = {trace.stats.channel[-1].upper() for trace in st_seismic}
if components == {"Z", "N", "E"}:
    st_zne = st_seismic.copy()
else:
    st_zne = st_seismic.copy()
    st_zne.rotate(method="->ZNE", inventory=inventory)

st_zne.merge(method=1, fill_value="interpolate")
st_zne.trim(
    max(trace.stats.starttime for trace in st_zne),
    min(trace.stats.endtime for trace in st_zne),
    pad=False,
)
st_zne.sort(keys=["channel"])


def unique_component(stream: Stream, component: str) -> Trace:
    matches = [
        trace for trace in stream
        if trace.stats.channel.upper().endswith(component.upper())
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one {component} component; found {[trace.id for trace in matches]}"
        )
    return matches[0]


tr_z = unique_component(st_zne, "Z")
tr_n = unique_component(st_zne, "N")
tr_e = unique_component(st_zne, "E")

sampling_rates = {float(trace.stats.sampling_rate) for trace in (tr_z, tr_n, tr_e)}
lengths = {int(trace.stats.npts) for trace in (tr_z, tr_n, tr_e)}
starts = {trace.stats.starttime for trace in (tr_z, tr_n, tr_e)}
if len(sampling_rates) != 1 or len(lengths) != 1 or len(starts) != 1:
    raise ValueError(
        "Z/N/E traces are not sample-aligned: "
        f"sampling_rates={sampling_rates}, lengths={lengths}, starts={starts}"
    )

back_azimuth_deg = float(
    analysis_config["array_reference"]["back_azimuth_to_slc40_deg"]
)
r_data, t_data = rotate_ne_rt(
    tr_n.data.astype(float),
    tr_e.data.astype(float),
    back_azimuth_deg,
)

tr_r = tr_n.copy()
tr_r.data = np.asarray(r_data, dtype=np.float64)
tr_r.stats.channel = tr_r.stats.channel[:-1] + "R"
tr_r.stats.processing = list(getattr(tr_r.stats, "processing", []))
tr_r.stats.processing.append(
    f"ObsPy NE->RT rotation; back_azimuth={back_azimuth_deg:.6f} deg"
)

tr_t = tr_e.copy()
tr_t.data = np.asarray(t_data, dtype=np.float64)
tr_t.stats.channel = tr_t.stats.channel[:-1] + "T"
tr_t.stats.processing = list(getattr(tr_t.stats, "processing", []))
tr_t.stats.processing.append(
    f"ObsPy NE->RT rotation; back_azimuth={back_azimuth_deg:.6f} deg"
)

st_zrt = Stream([tr_z.copy(), tr_r, tr_t])

with ROTATED_ZNE_PICKLE.open("wb") as file_object:
    pickle.dump(st_zne, file_object, protocol=pickle.HIGHEST_PROTOCOL)
with ROTATED_ZRT_PICKLE.open("wb") as file_object:
    pickle.dump(st_zrt, file_object, protocol=pickle.HIGHEST_PROTOCOL)

print("ZNE:", [trace.id for trace in st_zne])
print("ZRT:", [trace.id for trace in st_zrt])
print(f"SLC-40 back-azimuth: {back_azimuth_deg:.3f}°")


TypeError: __hash__ method should return an integer

## 4. Prepare the filtered analysis traces


In [ ]:
st_analysis = st_zne.copy()
st_analysis.detrend("linear")
st_analysis.detrend("demean")
st_analysis.taper(max_percentage=0.02, type="cosine")
st_analysis.filter(
    "bandpass",
    freqmin=FILTER_FREQMIN_HZ,
    freqmax=FILTER_FREQMAX_HZ,
    corners=FILTER_CORNERS,
    zerophase=FILTER_ZEROPHASE,
)

z = unique_component(st_analysis, "Z")
n = unique_component(st_analysis, "N")
e = unique_component(st_analysis, "E")

print(st_analysis)


## 5. Covariance linearity, rectilinearity, and planarity


In [ ]:
def covariance_polarization(
    z_values: np.ndarray,
    n_values: np.ndarray,
    e_values: np.ndarray,
    *,
    sampling_rate_hz: float,
    window_length_s: float,
    overlap: float,
):
    """Sliding real-covariance polarization in [Z, N, E] coordinates."""
    x = np.column_stack([
        np.asarray(z_values, dtype=float),
        np.asarray(n_values, dtype=float),
        np.asarray(e_values, dtype=float),
    ])
    nwin = max(3, int(round(window_length_s * sampling_rate_hz)))
    nstep = max(1, int(round(nwin * (1.0 - overlap))))

    rows = []
    for i0 in range(0, len(x) - nwin + 1, nstep):
        xx = x[i0:i0 + nwin]
        finite = np.isfinite(xx).all(axis=1)
        xx = xx[finite]
        center_sample = i0 + 0.5 * (nwin - 1)

        if len(xx) < 3:
            rows.append((center_sample, *([np.nan] * 8)))
            continue

        xx = xx - xx.mean(axis=0, keepdims=True)
        covariance = np.cov(xx, rowvar=False, bias=False)
        eigenvalues, eigenvectors = np.linalg.eigh(covariance)
        order = np.argsort(eigenvalues)[::-1]
        eigenvalues = np.maximum(eigenvalues[order], 0.0)
        eigenvectors = eigenvectors[:, order]
        lambda1, lambda2, lambda3 = eigenvalues

        if lambda1 <= np.finfo(float).eps:
            linearity = rectilinearity = planarity = np.nan
            azimuth_deg = incidence_deg = np.nan
        else:
            linearity = (lambda1 - lambda2) / lambda1
            rectilinearity = 1.0 - (lambda2 + lambda3) / (2.0 * lambda1)
            planarity = (lambda2 - lambda3) / lambda1

            vz, vn, ve = eigenvectors[:, 0]
            if vz < 0:
                vz, vn, ve = -vz, -vn, -ve
            incidence_deg = np.degrees(np.arctan2(np.hypot(vn, ve), abs(vz)))
            azimuth_deg = np.degrees(np.arctan2(ve, vn)) % 180.0

        rows.append((
            center_sample,
            linearity,
            rectilinearity,
            planarity,
            azimuth_deg,
            incidence_deg,
            lambda1,
            lambda2,
            lambda3,
        ))

    columns = [
        "center_sample", "linearity", "rectilinearity", "planarity",
        "covariance_azimuth_deg", "covariance_incidence_deg",
        "lambda1", "lambda2", "lambda3",
    ]
    return pd.DataFrame(rows, columns=columns), nwin, nstep


sampling_rate_hz = float(z.stats.sampling_rate)
covariance_df, covariance_nwin, covariance_nstep = covariance_polarization(
    z.data,
    n.data,
    e.data,
    sampling_rate_hz=sampling_rate_hz,
    window_length_s=WINDOW_LENGTH_S,
    overlap=WINDOW_OVERLAP,
)

covariance_df["time_epoch_s"] = (
    float(z.stats.starttime.timestamp)
    + covariance_df["center_sample"] / sampling_rate_hz
)
covariance_df["time_utc"] = covariance_df["time_epoch_s"].map(
    lambda value: str(UTCDateTime(value))
)
covariance_df["elapsed_from_explosion_s"] = (
    covariance_df["time_epoch_s"] - float(config.EXPLOSION_TIME.timestamp)
)

display(covariance_df.head())


## 6. TwistPy complex-signal ellipticity


In [ ]:
analysis = TimeDomainAnalysis3C(
    N=n,
    E=e,
    Z=z,
    window={
        "window_length_seconds": float(WINDOW_LENGTH_S),
        "overlap": float(WINDOW_OVERLAP),
    },
    timeaxis="utc",
)
analysis.polarization_analysis()

# TwistPy returns one value per analysis window. Convert its UTCDateTime list
# to epoch seconds so the output is portable and easy to merge/interpolate.
twistpy_epoch_s = np.asarray(
    [float(value.timestamp) for value in analysis.t_windows],
    dtype=float,
)

twistpy_df = pd.DataFrame({
    "time_epoch_s": twistpy_epoch_s,
    "time_utc": [str(value) for value in analysis.t_windows],
    "elapsed_from_explosion_s": twistpy_epoch_s - float(config.EXPLOSION_TIME.timestamp),
    "ellipticity": np.asarray(analysis.elli, dtype=float),
    "degree_of_polarization": np.asarray(analysis.dop, dtype=float),
    "twistpy_major_azimuth_deg": np.asarray(analysis.azi1, dtype=float),
    "twistpy_minor_azimuth_deg": np.asarray(analysis.azi2, dtype=float),
    "twistpy_major_inclination_deg": np.asarray(analysis.inc1, dtype=float),
    "twistpy_minor_inclination_deg": np.asarray(analysis.inc2, dtype=float),
})

display(twistpy_df.head())
print("Covariance windows:", len(covariance_df))
print("TwistPy windows:", len(twistpy_df))


## 7. Merge and write the canonical polarization products


In [ ]:
# The implementations can differ by a fraction of a sample in their window
# centers. Merge each covariance result onto the nearest TwistPy window with a
# tolerance of one covariance step.
merge_tolerance_s = max(
    covariance_nstep / sampling_rate_hz,
    1.0 / sampling_rate_hz,
)

polarization = pd.merge_asof(
    twistpy_df.sort_values("time_epoch_s"),
    covariance_df.drop(columns=["time_utc", "elapsed_from_explosion_s"]).sort_values("time_epoch_s"),
    on="time_epoch_s",
    direction="nearest",
    tolerance=merge_tolerance_s,
)

preferred_order = [
    "time_utc", "time_epoch_s", "elapsed_from_explosion_s",
    "linearity", "rectilinearity", "planarity", "ellipticity",
    "degree_of_polarization",
    "covariance_azimuth_deg", "covariance_incidence_deg",
    "twistpy_major_azimuth_deg", "twistpy_major_inclination_deg",
    "twistpy_minor_azimuth_deg", "twistpy_minor_inclination_deg",
    "lambda1", "lambda2", "lambda3", "center_sample",
]
polarization = polarization[[
    column for column in preferred_order if column in polarization.columns
]]

polarization.to_csv(POLARIZATION_CSV, index=False)
np.savez_compressed(
    POLARIZATION_NPZ,
    **{
        column: polarization[column].to_numpy()
        for column in polarization.columns
        if column != "time_utc"
    },
)

summary = {
    "producer_notebook": "12_rotate_and_compute_polarization.ipynb",
    "source_analysis_configuration": str(ANALYSIS_CONFIG_FILE),
    "source_waveform_stream": str(SOURCE_STREAM_FILE),
    "stationxml": str(STATIONXML_FILE),
    "seismic_channels": list(seismic_channels),
    "back_azimuth_to_slc40_deg": back_azimuth_deg,
    "filter": {
        "type": "bandpass",
        "freqmin_hz": FILTER_FREQMIN_HZ,
        "freqmax_hz": FILTER_FREQMAX_HZ,
        "corners": FILTER_CORNERS,
        "zerophase": FILTER_ZEROPHASE,
    },
    "window_length_s": WINDOW_LENGTH_S,
    "window_overlap": WINDOW_OVERLAP,
    "covariance_step_s": covariance_nstep / sampling_rate_hz,
    "definitions": {
        "linearity": "(lambda1 - lambda2) / lambda1",
        "rectilinearity": "1 - (lambda2 + lambda3) / (2 lambda1)",
        "planarity": "(lambda2 - lambda3) / lambda1",
        "ellipticity": "TwistPy complex-signal polarization ellipse attribute",
    },
    "outputs": {
        "polarization_csv": str(POLARIZATION_CSV),
        "polarization_npz": str(POLARIZATION_NPZ),
        "rotated_zne_pickle": str(ROTATED_ZNE_PICKLE),
        "rotated_zrt_pickle": str(ROTATED_ZRT_PICKLE),
        "diagnostic_figure": str(DIAGNOSTIC_FIGURE),
    },
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2) + "\n")

for path in (POLARIZATION_CSV, POLARIZATION_NPZ, ROTATED_ZNE_PICKLE, ROTATED_ZRT_PICKLE, SUMMARY_JSON):
    if not path.exists():
        raise FileNotFoundError(path)
    print(path)

display(polarization.head())


## 8. Diagnostic waveform and polarization figure


In [ ]:
plot_stream = st_analysis.copy().trim(PLOT_START, PLOT_END, pad=False)
plot_pol = polarization.loc[
    (polarization["time_epoch_s"] >= float(PLOT_START.timestamp))
    & (polarization["time_epoch_s"] <= float(PLOT_END.timestamp))
].copy()
plot_pol["plot_time_s"] = plot_pol["time_epoch_s"] - float(config.EXPLOSION_TIME.timestamp)

fig, axes = plt.subplots(5, 1, figsize=(12, 11), sharex=True, constrained_layout=True)

for ax, component in zip(axes[:3], ("Z", "N", "E")):
    trace = unique_component(plot_stream, component)
    times_s = trace.times(reftime=config.EXPLOSION_TIME)
    ax.plot(times_s, trace.data, color="black", lw=0.7)
    ax.set_ylabel(f"{component}\n(m s$^{{-1}}$)")
    ax.axvline(0.0, color="tab:red", lw=0.8, ls="--")

axes[3].plot(plot_pol["plot_time_s"], plot_pol["linearity"], label="Linearity")
axes[3].plot(plot_pol["plot_time_s"], plot_pol["rectilinearity"], label="Rectilinearity")
axes[3].plot(plot_pol["plot_time_s"], plot_pol["planarity"], label="Planarity")
axes[3].set_ylim(-0.05, 1.05)
axes[3].set_ylabel("Covariance\nattributes")
axes[3].legend(loc="upper right", ncol=3)

axes[4].plot(plot_pol["plot_time_s"], plot_pol["ellipticity"], label="Ellipticity")
axes[4].plot(plot_pol["plot_time_s"], plot_pol["degree_of_polarization"], label="TwistPy DOP")
axes[4].set_ylim(-0.05, 1.05)
axes[4].set_ylabel("TwistPy\nattributes")
axes[4].set_xlabel("Time relative to adopted explosion time (s)")
axes[4].legend(loc="upper right", ncol=2)

for ax in axes[3:]:
    ax.axvline(0.0, color="tab:red", lw=0.8, ls="--")
    ax.grid(True, alpha=0.2)

fig.suptitle(
    f"BCHH three-component polarization ({FILTER_FREQMIN_HZ:g}–{FILTER_FREQMAX_HZ:g} Hz; "
    f"{WINDOW_LENGTH_S:g} s window)"
)
fig.savefig(DIAGNOSTIC_FIGURE, dpi=180, bbox_inches="tight")
plt.show()

print("Diagnostic figure:", DIAGNOSTIC_FIGURE)


## 9. Authoritative outputs


In [ ]:
authoritative_outputs = {
    "polarization CSV": POLARIZATION_CSV,
    "polarization NPZ": POLARIZATION_NPZ,
    "rotated ZNE Stream": ROTATED_ZNE_PICKLE,
    "rotated ZRT Stream": ROTATED_ZRT_PICKLE,
    "processing summary": SUMMARY_JSON,
    "diagnostic figure": DIAGNOSTIC_FIGURE,
}
for label, path in authoritative_outputs.items():
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    print(f"{label}: {path}")
